In [95]:
import asyncio
import json
import re
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Final, Iterable

from curl_cffi import requests
from curl_cffi.requests import AsyncSession

In [ ]:
# =========================
# SCORESWAY CONFIG
# =========================

SCORESWAY_CODE: Final[str] = "ft1tiv1inq7v1sk3y9tv12yh5"


@dataclass(frozen=True)
class ScoreswaySeasonCode:
    season: str
    tmcl: str
    callback: str | None = None


@dataclass(frozen=True)
class ScoreswayLeague:
    league_name: str
    country_name: str
    slug: str
    seasons: dict[str, ScoreswaySeasonCode]


SCORESWAY_LEAGUES: dict[str, ScoreswayLeague] = {
    # =========================
    # ALBANIA
    # =========================
    "albania-superliga": ScoreswayLeague(
        league_name="Superliga",
        country_name="Albania",
        slug="albania-superliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="8yg6sm6qrfg00fr768chdw8wk",
                callback="W3393af3f377f0cde94ade76c7229a215e4a6a0775",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="cl624sknqv0xydwkvuz4jvpjo",
                callback="W3597a32ec897e785de0da2304cd9044917016159b",
            ),
        },
    ),

    # =========================
    # AUSTRIA
    # =========================
    "austria-bundesliga": ScoreswayLeague(
        league_name="Bundesliga",
        country_name="Austria",
        slug="austria-bundesliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="4jr49qim3gv5bjx3wjkf6n0us",
                callback="W3b95ecb241a36ce79a5b36e908370e529ef5cefae",
            ),
        },
    ),

    # =========================
    # BELGIUM
    # =========================
    "belgium-challenger-pro-league": ScoreswayLeague(
        league_name="Challenger Pro League",
        country_name="Belgium",
        slug="belgium-challenger-pro-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="25kyli40npy8mzwo9zij7zklw",
                callback="W32afeedab08e06a1b62fb3e255f339b64da692398",
            ),
        },
    ),

    "belgium-division-1": ScoreswayLeague(
        league_name="Division 1",
        country_name="Belgium",
        slug="belgium-division-1",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="b2h79dq31po18wufgvpd0ctuc",
                callback="W3b1f4c299a1f90cdbdc254733dfea1854869792df",
            ),
        },
    ),

    "belgium-jupiler-pro-league": ScoreswayLeague(
        league_name="Jupiler Pro League",
        country_name="Belgium",
        slug="belgium-jupiler-pro-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="16lkdpoccbh056r0v0bordo2c",
                callback="W3041cf8141858978467b51702547d959f4c2a0941",
            ),
        },
    ),

    "belgium-u18-elite": ScoreswayLeague(
        league_name="U18 Elite",
        country_name="Belgium",
        slug="belgium-u18-elite",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="cm6vcno46ld518o65wwj7ijh0",
                callback="W3d766125c36036f4970de9ff22d8d7160337da6ec",
            ),
        },
    ),

    # =========================
    # BOSNIA AND HERZEGOVINA
    # =========================
    "bosnia-and-herzegovina-first-league": ScoreswayLeague(
        league_name="First League",
        country_name="Bosnia and Herzegovina",
        slug="bosnia-and-herzegovina-first-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="675cfa32yfuxbeqg4am005h5g",
                callback="W3efb367365fa3ec71411688d23fbb45a4bb0df8b8",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="9s4z2xif1h92k16t27gyhdez8",
                callback="W355eaa56d147b4981cff19a670307b8e9d0012a2b",
            ),
        },
    ),

    "bosnia-and-herzegovina-premijer-liga": ScoreswayLeague(
        league_name="Premijer Liga",
        country_name="Bosnia and Herzegovina",
        slug="bosnia-and-herzegovina-premijer-liga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="f0lnv472mtq95zyt15gr72k9g",
                callback="W3039439ff412b91576fe48f46e5b385d501226ccf",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="3of502rzzgdxzicvws1qa3vv8",
                callback="W392c53fc8a385992f7d2a5036c03ed35f3662b3f8",
            ),
        },
    ),

    # =========================
    # BULGARIA
    # =========================
    "bulgaria-first-league": ScoreswayLeague(
        league_name="First League",
        country_name="Bulgaria",
        slug="bulgaria-first-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="45qbrg5zii9a6pj11cd916exg",
                callback="W3674b35491b5c3273ed227fd260206e8fd487e573",
            ),
        },
    ),

    "bulgaria-second-league": ScoreswayLeague(
        league_name="Second League",
        country_name="Bulgaria",
        slug="bulgaria-second-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="4qf9d0cy9w1nrrbwgnxi0d8us",
                callback="W35a48def1416ca14313a73e5aa9525ee4f2031221",
            ),
        },
    ),

    # =========================
    # CROATIA
    # =========================
    "croatia-first-nl": ScoreswayLeague(
        league_name="First NL",
        country_name="Croatia",
        slug="croatia-first-nl",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="d2gdw8kbigkmnpb9dw6y21wr8",
                callback="W34ac01a2bcfef9c4809fd161a29864090f6801e3b",
            ),
        },
    ),

    "croatia-hnl": ScoreswayLeague(
        league_name="HNL",
        country_name="Croatia",
        slug="croatia-hnl",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="am7k8a17kn4fzf3ty08y3hpg4",
                callback="W3a214644454426e9717c4f36a49a61b1a367c2cb9",
            ),
        },
    ),

    # =========================
    # CYPRUS
    # =========================
    "cyprus-1-division": ScoreswayLeague(
        league_name="1. Division",
        country_name="Cyprus",
        slug="cyprus-1-division",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="drabj8uucs4ooaz5mukhcvq50",
                callback="W38bc8cf2b17ee72a4829e5018e217309eff1363d5",
            ),
        },
    ),

    # =========================
    # CZECHIA
    # =========================
    "czechia-czech-liga": ScoreswayLeague(
        league_name="Czech Liga",
        country_name="Czechia",
        slug="czechia-czech-liga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="editpgeuef9yi9a8akjrl4bv8",
                callback="W3bcde9d1c7a36907c9a39df071500ab50123ce7b6",
            ),
        },
    ),

    # =========================
    # DENMARK
    # =========================
    "denmark-superliga": ScoreswayLeague(
        league_name="Superliga",
        country_name="Denmark",
        slug="denmark-superliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="3o8l1yf2irp018eaa2far455g",
                callback="W311fe72ea7a3dc39ede10ed5a36619ad97727f5d7",
            ),
        },
    ),

    "denmark-u17-ligaen": ScoreswayLeague(
        league_name="U17 Ligaen",
        country_name="Denmark",
        slug="denmark-u17-ligaen",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="b1nrrs5e7tude788ovvyehk44",
                callback="W3a8c6b203ec64e1b272b5de541bf615a5f6537c97",
            ),
        },
    ),

    "denmark-u19-ligaen": ScoreswayLeague(
        league_name="U19 Ligaen",
        country_name="Denmark",
        slug="denmark-u19-ligaen",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="33hbmhb874a6hevhb3nx1t8no",
                callback="W3b96f20b016844d96dc5ccba301ba0d9818ca89ee",
            ),
        },
    ),

    # =========================
    # ENGLAND
    # =========================
    "england-championship": ScoreswayLeague(
        league_name="Championship",
        country_name="England",
        slug="england-championship",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="bmmk637l2a33h90zlu36kx8no",
                callback="W36f8774befb0cac3a26a71d99b67e3cd1dd2c8d7b",
            ),
        },
    ),

    "england-league-one": ScoreswayLeague(
        league_name="League One",
        country_name="England",
        slug="england-league-one",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="5as0oqxeagsuzyf75xm7p5jbo",
                callback="W37fa4d04705f35ec72e01284c5ef4f21db358015b",
            ),
        },
    ),

    "england-league-two": ScoreswayLeague(
        league_name="League Two",
        country_name="England",
        slug="england-league-two",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="1fupi48urd1qk7t1rj12eyxw4",
                callback="W33d783dd75a6edfdcc8dbb9d1970e66585fe96727",
            ),
        },
    ),

    "england-national-league": ScoreswayLeague(
        league_name="National League",
        country_name="England",
        slug="england-national-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="ews1skrbpjw14ghhsw6h45lw4",
                callback="W35bc3a067fd0df8201e86663fedfef68b650f55f7",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="1odwnojqe6n8z0ftwkcwr4z6c",
                callback="W314a0752824ee300cb8bcb702cc64fd61d5878c39",
            ),
        },
    ),

    "england-premier-league": ScoreswayLeague(
        league_name="Premier League",
        country_name="England",
        slug="england-premier-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="51r6ph2woavlbbpk8f29nynf8",
                callback="W35a065b2dbc6542d92340e574c2e53ab1c324c862",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="9n12waklv005j8r32sfjj2eqc",
                callback="W3f1a3c26907fc232e65551a74fc12e2f7e98b01ae",
            ),
        },
    ),

    "england-premier-league-2": ScoreswayLeague(
        league_name="Premier League 2",
        country_name="England",
        slug="england-premier-league-2",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="cjzwab5sfnovticw96cr9yiac",
                callback="W3fab3ad7f01591f38e2477633ccb42145b6f19b85",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="e9awrqro4xgw1v6dztrru9dec",
                callback="W3cb8e6b65b8530e55b599e3c0da46d69024eba0f7",
            ),
        },
    ),

    "england-u18-premier-league": ScoreswayLeague(
        league_name="U18 Premier League",
        country_name="England",
        slug="england-u18-premier-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="598ncsty62ghibvsol819t3x0",
                callback="W3f1a3c26907fc232e65551a74fc12e2f7e98b01ae",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="ae4g9rkru7lngy6t71lg5p0yc",
                callback="W3746301b61ea4f59bb3522b7df77b077621adc6b3",
            ),
        },
    ),

    # =========================
    # ESTONIA
    # =========================
    "estonia-premium-liiga": ScoreswayLeague(
        league_name="Premium Liiga",
        country_name="Estonia",
        slug="estonia-premium-liiga",
        seasons={
            "2026": ScoreswaySeasonCode(
                season="2026",
                tmcl="co1q4zpn5ep76y2tiqohorspg",
                callback="W30afc4d83e97223936fef1bdaceca562e8d8bebef",
            ),
            "2025": ScoreswaySeasonCode(
                season="2025",
                tmcl="cun39cvobo7n76k9y2en7sglw",
                callback="W3edffd2c638179df5cafc80d8f19759e30d068a19",
            ),
        },
    ),

    # =========================
    # FRANCE
    # =========================
    "france-championnat-national-u19": ScoreswayLeague(
        league_name="Championnat National U19",
        country_name="France",
        slug="france-championnat-national-u19",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="4foa938bzelqeuu2qax634w0k",
                callback="W3edfe749a2cefae153dca2c92b3c3ae8651a3eaf0",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="c3ait9rt835k6buqdhb3vemtw",
                callback="W3171a5f04b9918a980d0370f1a6c038399f70c931",
            ),
        },
    ),

    "france-ligue-1": ScoreswayLeague(
        league_name="Ligue 1",
        country_name="France",
        slug="france-ligue-1",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="dbxs75cag7zyip5re0ppsanmc",
                callback="W3aad4ef8bb33ee8adaff851d0e0407c1e3f5666de",
            ),
        },
    ),

    "france-ligue-2": ScoreswayLeague(
        league_name="Ligue 2",
        country_name="France",
        slug="france-ligue-2",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="f12tw9djrb2bvlzrtg4hhme50",
                callback="W3a9e64df2db1e5c5948c5af9f231faac37ccabdf4",
            ),
        },
    ),

    "france-ligue-3": ScoreswayLeague(
        league_name="Ligue 3",
        country_name="France",
        slug="france-ligue-3",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="6rle937m7y4ktnwcredad5g5w",
                callback="W357f604c25d93a41ae6443abc9b3493c85256632a",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="cb8hus6uffk2zzxndht6lardg",
                callback="W34cb8ead031aa1c676deb1f423eceff6bb15874ee",
            ),
        },
    ),

    # =========================
    # GERMANY
    # =========================
    "germany-2-bundesliga": ScoreswayLeague(
        league_name="2. Bundesliga",
        country_name="Germany",
        slug="germany-2-bundesliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="3mjmai3idiul01s8jm0g7x6hg",
                callback="W3d5481b69632d236eaa295843016cdd9ec61905da",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="aaiuzjpvbqkch3l9w6mizhct0",
                callback="W36338869f6d776dedebc530652a2073ac88bd1e69",
            ),
        },
    ),

    "germany-3-liga": ScoreswayLeague(
        league_name="3. Liga",
        country_name="Germany",
        slug="germany-3-liga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="ce2peiekg738wn1w3xnwwu590",
                callback="W39c244b80c249099ad38ec461fdcd82c41dfd6476",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="r36pztzv4rtrzmxjv552im1g",
                callback="W3e4ca376bbdcd2ec257fd87f1f2410595bc3f68ed",
            ),
        },
    ),

    "germany-bundesliga": ScoreswayLeague(
        league_name="Bundesliga",
        country_name="Germany",
        slug="germany-bundesliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="2bchmrj23l9u42d68ntcekob8",
                callback="W36dc9b454e86e1c480a643471228e6bc4dd595971",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="73zebisnu1109jix9yoc09yc4",
                callback="W35cdd53f6995668b4f6af428b15d58d0e0ad2e8e3",
            ),
        },
    ),

    "germany-u19-bundesliga": ScoreswayLeague(
        league_name="U19 Bundesliga",
        country_name="Germany",
        slug="germany-u19-bundesliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="7v37llgtguvvf86ctpptbzrpw",
                callback="W355cef47b44b7b1e77a6fc83929f3122f89e12c7e",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="ax6zdxusks2i0klwshc6mfl04",
                callback="W3b5bd5c7054cd0f0fc1d755792b23996bda3de9b3",
            ),
        },
    ),

    # =========================
    # ITALY
    # =========================
    "italy-serie-a": ScoreswayLeague(
        league_name="Serie A",
        country_name="Italy",
        slug="italy-serie-a",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="emdmtfr1v8rey2qru3xzfwges",
                callback="W3074b6fcd79e56b49eda6e30417b4a31d4a87e65d",
            ),
        },
    ),

    "italy-serie-b": ScoreswayLeague(
        league_name="Serie B",
        country_name="Italy",
        slug="italy-serie-b",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="1yx2eq0yvay996e4h7s0mraxg",
                callback="W36396dc9b75ac7309473220d9754c2dc616656725",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="6evrk16rq1mhn0eajdk0bvehg",
                callback="W3de1e1e8f71eff4b0e128bc2c4d7bfdf36e212df1",
            ),
        },
    ),

    # =========================
    # MONTENEGRO
    # =========================
    "montenegro-first-league": ScoreswayLeague(
        league_name="First League",
        country_name="Montenegro",
        slug="montenegro-first-league",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="bc7aji68rjakal72ovhprdzx0",
                callback="W3a8acc1bc26efd9814b6f6d16b247746bd2655d0e",
            ),
        },
    ),

    # =========================
    # NETHERLANDS
    # =========================
    "netherlands-eerste-divisie": ScoreswayLeague(
        league_name="Eerste Divisie",
        country_name="Netherlands",
        slug="netherlands-eerste-divisie",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="25u7u2cp66kftzrrneazqp3x0",
                callback="W3532cdb23e419cf470d4449bff5f676fa49f02df7",
            ),
        },
    ),

    "netherlands-eredivisie": ScoreswayLeague(
        league_name="Eredivisie",
        country_name="Netherlands",
        slug="netherlands-eredivisie",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="aouykkl1rt7zo06sg0kbzkbh0",
                callback="W32bd64268dbb2b5501cfc9982fc8b1159d6e1bd0a",
            ),
        },
    ),

    "netherlands-tweede-divisie": ScoreswayLeague(
        league_name="Tweede Divisie",
        country_name="Netherlands",
        slug="netherlands-tweede-divisie",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="bk2rzcvvw5kj6zw2ycdisglqs",
                callback="W3ec726c31ab6dfe34dd33f7f3672340ad0f53fbab",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="bo5ymeqyjxx2sc6odlkmu1udw",
                callback="W3a847def5d365e44d6e84ff1b56460e348548ac3a",
            ),
        },
    ),

    # =========================
    # POLAND
    # =========================
    "poland-ekstraklasa": ScoreswayLeague(
        league_name="Ekstraklasa",
        country_name="Poland",
        slug="poland-ekstraklasa",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="3ghh5adz62ws55se0snozmej8",
                callback="W3452d1b3bf899eaa81baa95866ca0411f17339201",
            ),
        },
    ),

    # =========================
    # SERBIA
    # =========================
    "serbia-prva-liga": ScoreswayLeague(
        league_name="Prva Liga",
        country_name="Serbia",
        slug="serbia-prva-liga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="54jd6j2c3t5uksf8s40khsh04",
                callback="W3057af742a38c6ff3229509aee2338d230c3484ae",
            ),
        },
    ),

    "serbia-superliga": ScoreswayLeague(
        league_name="SuperLiga",
        country_name="Serbia",
        slug="serbia-superliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="77y4osmpjr44rl7enktiv9qms",
                callback="W371397dbf3bb487d90c55aa68372ab04baeaf2bb7",
            ),
        },
    ),

    # =========================
    # SPAIN
    # =========================
    "spain-laliga": ScoreswayLeague(
        league_name="LaLiga",
        country_name="Spain",
        slug="spain-laliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="80zg2v1cuqcfhphn56u4qpyqc",
                callback="W32e1ce8ead468893ac276dde18d30e28441247d50",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="4x7uzww3jur4re7sgt3mslyj8",
                callback="W3e9d468d5a6e44c833aac9d1d8a2d775d10eeb4ba",
            ),
        },
    ),

    "spain-segunda-division": ScoreswayLeague(
        league_name="Segunda Division",
        country_name="Spain",
        slug="spain-segunda-division",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="dko0hzifl1xv9c51s3ai017v8",
                callback="W39ca4dfbadb4300670c31585b86f8b59e36672450",
            ),
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="9991qneanhxxxo8ioi14pw7bo",
                callback="W3a071492e13246062dab19152611199a14b73bda6",
            ),
        },
    ),
}

def get_scoresway_codes(
    league_slug: str,
    season: str,
) -> ScoreswaySeasonCode:
    try:
        league = SCORESWAY_LEAGUES[league_slug]
    except KeyError:
        raise ValueError(f"Unknown league slug: {league_slug}")

    try:
        return league.seasons[season]
    except KeyError:
        available = ", ".join(league.seasons.keys())
        raise ValueError(
            f"Season '{season}' not available for {league.league_name}. "
            f"Available seasons: {available}"
        )


# =========================
# SHARED HELPERS
# =========================

def parse_jsonp(text: str, callback: str) -> dict[str, Any] | list[Any]:
    text = text.strip()

    match = re.match(
        rf"^{re.escape(callback)}\((.*)\);?$",
        text,
        re.DOTALL,
    )

    if not match:
        raise ValueError(f"Invalid JSONP response:\n{text[:500]}")

    return json.loads(match.group(1))


def generate_callback() -> str:
    """
    Generates a callback similar to the ones used by the API.
    Example:
    W36c4b1298313d17bccab9c03f2cf1044819cf3b90
    """
    return f"W{uuid.uuid4().hex}{uuid.uuid4().hex[:8]}"


def get_scoresway_headers() -> dict[str, str]:
    return {
        "accept": "*/*",
        "accept-language": "en-GB,en;q=0.9",
        "referer": "https://www.scoresway.com/",
        "origin": "https://www.scoresway.com",
        "user-agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) "
            "Gecko/20100101 Firefox/150.0"
        ),
        "sec-fetch-dest": "script",
        "sec-fetch-mode": "no-cors",
        "sec-fetch-site": "cross-site",
    }


def slugify_league_name(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


# =========================
# FETCH SQUAD / TEAMS
# =========================

def fetch_scoresway_squad(
    *,
    scoresway_code: str,
    tmcl: str,
    callback: str | None = None,
    page_size: int = 200,
    timeout: int = 30,
    verbose: bool = True,
) -> dict[str, Any] | list[Any]:
    """
    Fetches the squad/team list for one Scoresway tournament calendar.

    This should be called before collect_scoresway_team_stats().
    """

    callback = callback or generate_callback()

    url = f"https://api.performfeeds.com/soccerdata/squads/{scoresway_code}/"

    params = {
        "_rt": "c",
        "tmcl": tmcl,
        "_pgSz": str(page_size),
        "_lcl": "en",
        "_fmt": "jsonp",
        "sps": "widgets",
        "_clbk": callback,
    }

    response = requests.get(
        url,
        params=params,
        headers=get_scoresway_headers(),
        impersonate="chrome120",
        timeout=timeout,
    )

    if verbose:
        print(f"Squad request | status={response.status_code}")
        print(response.text[:500])

    if response.status_code >= 400:
        raise RuntimeError(
            f"Squad request failed with HTTP {response.status_code}: "
            f"{response.text[:500]}"
        )

    return parse_jsonp(response.text, callback)


def extract_squad_teams(
    squad_data: dict[str, Any] | list[Any],
) -> list[dict[str, Any]]:
    """
    Extracts the real team list from the Scoresway squad response.

    The API response shape can vary, so this function handles the common cases.
    If the structure is different, inspect squad_data once and adjust here.
    """

    if isinstance(squad_data, list):
        return squad_data

    if not isinstance(squad_data, dict):
        raise TypeError(f"Unexpected squad_data type: {type(squad_data)}")

    possible_keys = [
        "squad",
        "squads",
        "contestants",
        "teams",
        "data",
    ]

    for key in possible_keys:
        value = squad_data.get(key)

        if isinstance(value, list):
            return value

        if isinstance(value, dict):
            nested = extract_squad_teams(value)
            if nested:
                return nested

    raise ValueError(
        "Could not find a team list inside squad_data. "
        "Inspect squad_data.keys() and update extract_squad_teams()."
    )


# =========================
# FETCH TEAM SEASON STATS
# =========================

async def fetch_team_season_stats(
    session: AsyncSession,
    *,
    url: str,
    headers: dict[str, str],
    team: dict[str, Any],
    timeout: int = 30,
    max_retries: int = 2,
    retry_delay: float = 1.0,
) -> dict[str, Any]:
    ctst = team.get("contestantId")
    tmcl = team.get("tournamentCalendarId")
    contestant_name = team.get("contestantName", ctst)

    if not ctst or not tmcl:
        return {
            "contestantId": ctst,
            "contestantName": contestant_name,
            "tournamentCalendarId": tmcl,
            "error": f"Invalid team object: {team}",
        }

    for attempt in range(1, max_retries + 2):
        callback = generate_callback()

        params = {
            "_rt": "c",
            "tmcl": tmcl,
            "ctst": ctst,
            "_lcl": "en",
            "_fmt": "jsonp",
            "sps": "widgets",
            "_clbk": callback,
        }

        try:
            response = await session.get(
                url,
                params=params,
                headers=headers,
                impersonate="chrome120",
                timeout=timeout,
            )

            print(f"{contestant_name} | status={response.status_code}")

            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:300]}"
                )

            data = parse_jsonp(response.text, callback)

            return {
                "contestantId": ctst,
                "contestantName": contestant_name,
                "tournamentCalendarId": tmcl,
                "data": data,
            }

        except Exception as exc:
            if attempt <= max_retries:
                await asyncio.sleep(retry_delay * attempt)
                continue

            print(f"Failed for {contestant_name}: {exc}")

            return {
                "contestantId": ctst,
                "contestantName": contestant_name,
                "tournamentCalendarId": tmcl,
                "error": str(exc),
            }


async def collect_scoresway_team_stats(
    teams: Iterable[dict[str, Any]],
    *,
    scoresway_code: str,
    max_concurrent: int = 5,
    timeout: int = 30,
    max_retries: int = 2,
    retry_delay: float = 1.0,
) -> list[dict[str, Any]]:
    """
    Collects season stats for every team in a squad list.

    Recommended starting point:
    max_concurrent=4 or 5.
    """

    url = f"https://api.performfeeds.com/soccerdata/seasonstats/{scoresway_code}/"

    headers = get_scoresway_headers()
    semaphore = asyncio.Semaphore(max_concurrent)

    async with AsyncSession(max_clients=max_concurrent) as session:

        async def bounded_fetch(team: dict[str, Any]) -> dict[str, Any]:
            async with semaphore:
                return await fetch_team_season_stats(
                    session,
                    url=url,
                    headers=headers,
                    team=team,
                    timeout=timeout,
                    max_retries=max_retries,
                    retry_delay=retry_delay,
                )

        tasks = [bounded_fetch(team) for team in teams]
        results = await asyncio.gather(*tasks)

    print(f"Collected {len(results)} results")

    return results


# ============================
# BUILD TEAM & PLAYER STATS DF
# =============================
import pandas as pd


def build_team_stats_df(result_list: list[dict]) -> pd.DataFrame:
    rows = []

    for result in result_list:
        data = result.get("data", {})

        contestant = data.get("contestant")
        competition = data.get("competition", {})
        tournament_calendar = data.get("tournamentCalendar", {})

        if not contestant:
            continue

        contestant_id = contestant.get("id")

        row = {
            # Contestant / team
            "contestant_id": contestant_id,
            "contestant_name": contestant.get("name"),

            # Team badge URLs
            "badge_sm": (
                "https://omo.akamai.opta.net/image.php"
                f"?h=www.scoresway.com&sport=football&entity=team"
                f"&description=badges&dimensions=65&id={contestant_id}"
            ),
            "badge_lg": (
                "https://omo.akamai.opta.net/image.php"
                f"?h=www.scoresway.com&sport=football&entity=team"
                f"&description=badges&dimensions=150&id={contestant_id}"
            ),

            # Competition
            "competition_id": competition.get("id"),
            "competition_name": competition.get("name"),
            "competition_known_name": competition.get("knownName"),

            # Season / tournament calendar
            "tournament_calendar_id": tournament_calendar.get("id"),
            "season_name": tournament_calendar.get("name"),
            "season_start_date": tournament_calendar.get("startDate"),
            "season_end_date": tournament_calendar.get("endDate"),
        }

        for stat in contestant.get("stat", []):
            stat_name = stat.get("name")
            stat_value = stat.get("value")

            if stat_name:
                row[stat_name.strip()] = stat_value

        rows.append(row)

    df = pd.DataFrame(rows)

    metadata_columns = {
        "contestant_id",
        "contestant_name",
        "badge_sm",
        "badge_lg",
        "competition_id",
        "competition_name",
        "competition_known_name",
        "tournament_calendar_id",
        "season_name",
        "season_start_date",
        "season_end_date",
    }

    for col in df.columns:
        if col not in metadata_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def build_player_stats_df(result_list: list[dict]) -> pd.DataFrame:
    rows = []

    for result in result_list:
        data = result.get("data", {})

        competition = data.get("competition", {})
        tournament_calendar = data.get("tournamentCalendar", {})
        contestant = data.get("contestant", {})

        players = data.get("player", [])

        if not players:
            continue

        for player in players:
            player_id = player.get("id")

            row = {
                # Competition
                "competition_id": competition.get("id"),
                "competition_name": competition.get("name"),
                "competition_known_name": competition.get("knownName"),

                # Season
                "tournament_calendar_id": tournament_calendar.get("id"),
                "season_name": tournament_calendar.get("name"),
                "season_start_date": tournament_calendar.get("startDate"),
                "season_end_date": tournament_calendar.get("endDate"),

                # Team / contestant
                "contestant_id": contestant.get("id"),
                "contestant_name": contestant.get("name"),

                # Player
                "player_id": player_id,
                "player_image": (
                    "https://omo.akamai.opta.net/image.php"
                    "?secure=true&h=omo.akamai.opta.net"
                    "&sport=football&entity=player"
                    "&description=6eqit8ye8aomdsrrq0hk3v7gh"
                    f"&dimensions=103x155&id={player_id}"
                ),
                "shirt_number": player.get("shirtNumber"),
                "first_name": player.get("firstName"),
                "last_name": player.get("lastName"),
                "short_first_name": player.get("shortFirstName"),
                "short_last_name": player.get("shortLastName"),
                "match_name": player.get("matchName"),
                "position": player.get("position"),
            }

            for stat in player.get("stat", []):
                stat_name = stat.get("name")
                stat_value = stat.get("value")

                if stat_name:
                    row[stat_name.strip()] = stat_value

            rows.append(row)

    df = pd.DataFrame(rows)

    metadata_columns = {
        "competition_id",
        "competition_name",
        "competition_known_name",
        "tournament_calendar_id",
        "season_name",
        "season_start_date",
        "season_end_date",
        "contestant_id",
        "contestant_name",
        "player_id",
        "player_image",
        "shirt_number",
        "first_name",
        "last_name",
        "short_first_name",
        "short_last_name",
        "match_name",
        "position",
    }

    for col in df.columns:
        if col not in metadata_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

# =========================
# SAVE CSV FILES
# =========================

def save_scoresway_stats_csvs(
    *,
    team_stats_df,
    player_stats_df,
    cur_league: str,
    output_dir: str | Path = "data/processed",
    encoding: str = "utf-8",
) -> dict[str, Path]:
    """
    Saves team and player stats CSV files for one league.

    Example:
    data/processed/netherlands_eredivisie_team_stats.csv
    data/processed/netherlands_eredivisie_player_stats.csv
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    league_prefix = slugify_league_name(cur_league)

    team_stats_path = output_dir / f"{league_prefix}_team_stats.csv"
    player_stats_path = output_dir / f"{league_prefix}_player_stats.csv"

    team_stats_df.to_csv(
        team_stats_path,
        index=False,
        encoding=encoding,
    )

    player_stats_df.to_csv(
        player_stats_path,
        index=False,
        encoding=encoding,
    )

    print("CSV files saved successfully:")
    print(team_stats_path)
    print(player_stats_path)

    return {
        "team_stats": team_stats_path,
        "player_stats": player_stats_path,
    }






In [97]:
async def scrape_scoresway_league_season(
    *,
    league_slug: str,
    season: str,
    scoresway_code: str = SCORESWAY_CODE,
    max_concurrent: int = 5,
    output_dir: str | Path = "data/processed",
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Runs the full Scoresway scraping flow for one league-season.

    Flow:
    1. Get league config
    2. Fetch squad/team list
    3. Collect team season stats
    4. Build team/player DataFrames
    5. Save CSV files
    """

    league = SCORESWAY_LEAGUES[league_slug]
    codes = get_scoresway_codes(league_slug, season)

    cur_league = f"{league.slug}_{season}"

    if verbose:
        print("=" * 80)
        print(f"Scraping: {league.country_name} - {league.league_name} - {season}")
        print("=" * 80)

    squad_data = fetch_scoresway_squad(
        scoresway_code=scoresway_code,
        tmcl=codes.tmcl,
        callback=codes.callback,
        verbose=verbose,
    )

    squad = extract_squad_teams(squad_data)

    if verbose:
        print(f"Teams found: {len(squad)}")

    result_list = await collect_scoresway_team_stats(
        squad,
        scoresway_code=scoresway_code,
        max_concurrent=max_concurrent,
    )

    team_stats_df = build_team_stats_df(result_list)
    player_stats_df = build_player_stats_df(result_list)

    saved_paths = save_scoresway_stats_csvs(
        team_stats_df=team_stats_df,
        player_stats_df=player_stats_df,
        cur_league=cur_league,
        output_dir=output_dir,
    )

    return {
        "league_slug": league_slug,
        "league_name": league.league_name,
        "country_name": league.country_name,
        "season": season,
        "teams_count": len(squad),
        "results_count": len(result_list),
        "team_stats_rows": len(team_stats_df),
        "player_stats_rows": len(player_stats_df),
        "saved_paths": saved_paths,
        "errors": [
            result for result in result_list
            if "error" in result
        ],
    }

async def scrape_all_scoresway_leagues(
    *,
    scoresway_leagues: dict[str, ScoreswayLeague] = SCORESWAY_LEAGUES,
    scoresway_code: str = SCORESWAY_CODE,
    max_concurrent: int = 5,
    output_dir: str | Path = "data/processed",
    delay_between_league_seasons: float = 2.0,
    continue_on_error: bool = True,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    """
    Iterates through every league and every season inside SCORESWAY_LEAGUES.

    It runs league-seasons sequentially, but each league-season fetches teams
    concurrently through collect_scoresway_team_stats().
    """

    all_results = []

    for league_slug, league in scoresway_leagues.items():
        for season in league.seasons.keys():
            try:
                result = await scrape_scoresway_league_season(
                    league_slug=league_slug,
                    season=season,
                    scoresway_code=scoresway_code,
                    max_concurrent=max_concurrent,
                    output_dir=output_dir,
                    verbose=verbose,
                )

                all_results.append(result)

                if verbose:
                    print(
                        f"Finished {league.league_name} {season}: "
                        f"{result['team_stats_rows']} team rows, "
                        f"{result['player_stats_rows']} player rows"
                    )

            except Exception as exc:
                error_result = {
                    "league_slug": league_slug,
                    "league_name": league.league_name,
                    "country_name": league.country_name,
                    "season": season,
                    "error": str(exc),
                }

                all_results.append(error_result)

                print(
                    f"Failed {league.country_name} - "
                    f"{league.league_name} - {season}: {exc}"
                )

                if not continue_on_error:
                    raise

            if delay_between_league_seasons > 0:
                await asyncio.sleep(delay_between_league_seasons)

    print("=" * 80)
    print(f"Completed scraping {len(all_results)} league-season entries")
    print("=" * 80)

    return all_results

In [ ]:
all_scrape_results = await scrape_all_scoresway_leagues(
    max_concurrent=5,
    output_dir="data/processed",
    delay_between_league_seasons=2.0,
    continue_on_error=True,
    verbose=True,
)